# PolyWin R2 — v14: FULL-PI1M MT-GNN pretraining (P14 experiment)

## Experiment (pre-registered)
* v13 baseline scored **0.877 LB** with a PI1M pretrain budget of 20k molecules, 5 epochs.
* **P14 changes ONLY pretraining scale:** the GINE encoder is now pretrained on the
  **entire PI1M archive** (~995k deduped molecules) with a **larger budget** (10 epochs).
* The downstream MT-GNN, GBM trio stack, per-target Ridge blend, descriptors,
  folds, and all three GNN seeds (42,999,2025) are **BIT-IDENTICAL** to the
  frozen v13 baseline.

## Success / fail criteria (judged only after the Kaggle run)
* **Pass A:** corr(GBM OOF, MT-GNN OOF) per target drops from ~0.915–0.969 toward ~0.88–0.92
  (the GNN learned genuinely new chemistry).
* **Pass B:** blend OOF gain ≥ +0.005 over the v13 baseline.
* **Fail:** correlation stays ~0.95+ AND blend OOF gain < 0.005 → freeze the
  pretraining line, no more pretrain tuning.

Only OSI-approved libs: PyTorch, PyG, RDKit, scikit-learn, LightGBM, CatBoost, XGBoost.


In [ ]:
import os, sys, time, gc, random, warnings
import subprocess, importlib.util

def ensure_pkg(pkg, import_name=None):
    name = import_name or pkg
    if importlib.util.find_spec(name) is None:
        print("installing", pkg, flush=True)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--disable-pip-version-check", pkg])

for _p, _n in [("rdkit", "rdkit"), ("torch_geometric", "torch_geometric"),
               ("lightgbm", "lightgbm"), ("catboost", "catboost"), ("xgboost", "xgboost"),
               ("scipy", "scipy")]:
    ensure_pkg(_p, _n)

# --- CUDA probe / repair identical to v13 (P100 sm_60) ---
_probe = ('import torch;' + 'a=torch.zeros(4,device="cuda");b=a+1;torch.cuda.synchronize();print("OK")')
def _force_cuda():
    try:
        _r = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                            text=True, timeout=600)
    except Exception:
        _r = None
    if _r is not None and _r.returncode == 0 and "OK" in (_r.stdout or ""):
        return
    print("CUDA kernel missing; installing torch 2.5.1 (cu121, supports P100 sm_60)...", flush=True)
    try:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                               "--no-cache-dir", "--index-url",
                               "https://download.pytorch.org/whl/cu121", "torch==2.5.1"],
                              timeout=1800)
        _r2 = subprocess.run([sys.executable, "-c", _probe], capture_output=True,
                             text=True, timeout=600)
        print("post-reinstall probe rc:", _r2.returncode, flush=True)
    except Exception as _e:
        print("torch reinstall errored:", repr(_e)[:200], flush=True)
if os.path.exists("/kaggle"):
    _force_cuda()

import numpy as np
import pandas as pd
warnings.filterwarnings("ignore")
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.data import Data, Batch
from torch_geometric.nn import GINEConv, global_mean_pool, global_add_pool
from rdkit import Chem
from rdkit import RDLogger
RDLogger.DisableLog("rdApp.*")
from rdkit.Chem import Descriptors, AllChem, MACCSkeys, rdMolDescriptors, Crippen, GraphDescriptors
from rdkit.Chem import rdFingerprintGenerator
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold, train_test_split
from sklearn.preprocessing import StandardScaler
import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostRegressor

SMOKE = os.environ.get("SMOKE", "0") == "1"
SEED = 42
GNN_SEEDS = os.environ.get("GNN_SEEDS", "42,999,2025")
os.environ["GNN_SEEDS"] = GNN_SEEDS
print("GNN_SEEDS =", GNN_SEEDS, flush=True)
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
def _cuda_ok():
    if not torch.cuda.is_available():
        return False
    try:
        a = torch.zeros(4, device="cuda"); b = a + 1; torch.cuda.synchronize(); del a, b
        return True
    except Exception:
        return False
DEVICE = "cuda" if _cuda_ok() else "cpu"
print("device:", DEVICE, flush=True)
GLOBAL_FOLDS = 5
MAX_EPOCHS = 120
PATIENCE = 20
EARLY_HOLDOUT = 0.15
BS = 256
LR = 1e-3
if DEVICE == "cpu":
    GLOBAL_FOLDS = min(GLOBAL_FOLDS, 2)
    MAX_EPOCHS = min(MAX_EPOCHS, 12)
    BS = 256
    PATIENCE = min(PATIENCE, 6)
PRETRAIN_EPOCHS = 10
PRETRAIN_SAMPLE = 2000000

if os.path.exists("/kaggle"):
    WORK = "/kaggle/working"; INP = "/kaggle/input"
else:
    WORK = os.path.join("vault", "pipeline_out_v14")
    INP = "official_dataset"
os.makedirs(WORK, exist_ok=True)
PRETRAINED = os.path.join(WORK, "pretrained_encoder.pt")
OUT = WORK

print("----- P14 CONFIG -----", flush=True)
print("PRETRAIN_SAMPLE =", PRETRAIN_SAMPLE, "| PRETRAIN_EPOCHS =", PRETRAIN_EPOCHS, flush=True)
print("device:", DEVICE, "| SMOKE:", SMOKE, "| folds:", GLOBAL_FOLDS,
      "| out:", OUT, flush=True)


In [ ]:
from sklearn.linear_model import Ridge

TARGETS = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]
TARGET_IDX = {t: i for i, t in enumerate(TARGETS)}


## 1. Data — current-round CSVs, canonicalize, compute descriptors + fingerprints

In [ ]:
def find_input(base, name):
    for p in [os.path.join(base, name), os.path.join(base, "ppp-round-2", name),
              os.path.join(base, "competitions", "ppp-round-2", name)]:
        if os.path.exists(p):
            return p
    return None

def canonical(s):
    if not isinstance(s, str):
        return None, None, None
    m = Chem.MolFromSmiles(s)
    if m is None:
        return None, None, None
    try:
        c = Chem.MolToSmiles(m)
        ik = Chem.MolToInchiKey(m)
    except Exception:
        return Chem.MolToSmiles(m), None, None
    return c, ik, m

def canon_fast(s):
    """MolToSmiles only (identical string to canonical(s)[0]) for the PI1M
    pretrain corpus — avoids 995k MolToInchiKey computations, no result change."""
    if not isinstance(s, str):
        return None
    try:
        m = Chem.MolFromSmiles(s)
        return Chem.MolToSmiles(m) if m is not None else None
    except Exception:
        return None

def feats(m):
    if m is None:
        return [np.nan] * 35
    # Gasteiger partial charges
    try:
        Chem.rdPartialCharges.ComputeGasteigerCharges(m)
        gasteiger = [a.GetDoubleProp('_GasteigerCharge') for a in m.GetAtoms()]
        g_mean = np.mean(gasteiger)
        g_std = np.std(gasteiger) if len(gasteiger) > 1 else 0.0
        g_min = np.min(gasteiger); g_max = np.max(gasteiger)
    except Exception:
        g_mean = g_std = g_min = g_max = 0.0
    # Element composition
    atoms = m.GetAtoms()
    n_total = len(atoms) if atoms else 1
    elem_counts = {}
    for a in atoms:
        sym = a.GetSymbol()
        elem_counts[sym] = elem_counts.get(sym, 0) + 1
    frac_C = elem_counts.get("C", 0) / n_total
    frac_N = elem_counts.get("N", 0) / n_total
    frac_O = elem_counts.get("O", 0) / n_total
    frac_S = elem_counts.get("S", 0) / n_total
    frac_F = elem_counts.get("F", 0) / n_total
    n_hetero = sum(v for k, v in elem_counts.items() if k not in ("C", "H"))
    frac_hetero = n_hetero / n_total
    # Bond type ratios
    bonds = m.GetBonds()
    n_bonds = len(bonds) if bonds else 1
    bond_counts = {"SINGLE": 0, "DOUBLE": 0, "TRIPLE": 0, "AROMATIC": 0}
    for b in bonds:
        bt = b.GetBondType().name
        if bt in bond_counts:
            bond_counts[bt] += 1
    ratio_single = bond_counts["SINGLE"] / n_bonds
    ratio_double = bond_counts["DOUBLE"] / n_bonds
    ratio_aromatic = bond_counts["AROMATIC"] / n_bonds
    return [
        Descriptors.MolWt(m), Descriptors.MolLogP(m), Descriptors.TPSA(m),
        Descriptors.NumHDonors(m), Descriptors.NumHAcceptors(m),
        Descriptors.RingCount(m), Descriptors.NumAromaticRings(m),
        Descriptors.NumAliphaticRings(m), Descriptors.NumSaturatedRings(m),
        Descriptors.NumRotatableBonds(m), rdMolDescriptors.CalcNumHeavyAtoms(m),
        Descriptors.NumHeteroatoms(m), Descriptors.FractionCSP3(m),
        Crippen.MolMR(m), rdMolDescriptors.CalcNumBridgeheadAtoms(m),
        rdMolDescriptors.CalcNumSpiroAtoms(m),
        rdMolDescriptors.CalcNumAromaticAtoms(m) if hasattr(rdMolDescriptors, "CalcNumAromaticAtoms") else Descriptors.NumAromaticRings(m),
        GraphDescriptors.BalabanJ(m), GraphDescriptors.Ipc(m),
        rdMolDescriptors.CalcNumLipinskiHBA(m), rdMolDescriptors.CalcNumLipinskiHBD(m),
        rdMolDescriptors.CalcNumAtomStereoCenters(m),
        g_mean, g_std, g_min, g_max,
        frac_C, frac_N, frac_O, frac_S, frac_F, frac_hetero,
        ratio_single, ratio_double, ratio_aromatic,
    ]

FNAMES = ["MolWt", "LogP", "TPSA", "HDon", "HAccep", "RingCnt", "AroRing", "AliRing", "SatRing",
          "RotB", "HeavyAt", "HeteroAt", "FracCSP3", "MR", "Bridge", "Spiro", "AroAt",
          "BalabanJ", "Ipc", "LipHBA", "LIHBD", "Stereo",
          "GMean", "GStd", "GMin", "GMax",
          "FracC", "FracN", "FracO", "FracS", "FracF", "FracHetero",
          "RatioSingle", "RatioDouble", "RatioAro"]
assert len(FNAMES) == 35

train_path = find_input(INP, "train.csv")
test_path = find_input(INP, "test.csv")
assert train_path and test_path, "train.csv / test.csv not found in " + INP

tr = pd.read_csv(train_path)
te = pd.read_csv(test_path)
print("train:", tr.shape, "test:", te.shape, flush=True)

tcpl = tr["smiles"].map(canonical)
tr["canon"], tr["inchikey"], _ = zip(*tcpl)
tepl = te["smiles"].map(canonical)
te["canon"], te["inchikey"], _ = zip(*tepl)

tr_f = np.array(tr["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
te_f = np.array(te["smiles"].map(lambda s: feats(Chem.MolFromSmiles(s))).tolist())
tr[FNAMES] = tr_f
te[FNAMES] = te_f
print("descriptors done", flush=True)

trf = tr.dropna(subset=["target"]).copy()
tef = te.copy()

FEAT_COLS = [c for c in trf.columns if c not in
            ("smiles", "target", "target_type", "canon", "inchikey", "id")]
print("FEAT_COLS:", len(FEAT_COLS), flush=True)


In [ ]:
def add_fingerprints(df):
    morgan = np.zeros((len(df), 2048), dtype=np.float32)
    maccs = np.zeros((len(df), 167), dtype=np.float32)
    ap = np.zeros((len(df), 1024), dtype=np.float32)
    tt = np.zeros((len(df), 1024), dtype=np.float32)
    ap_gen = rdFingerprintGenerator.GetAtomPairGenerator(fpSize=1024)
    tt_gen = rdFingerprintGenerator.GetTopologicalTorsionGenerator(fpSize=1024)
    for i, s in enumerate(df["smiles"]):
        m = Chem.MolFromSmiles(s)
        if m is None:
            continue
        morgan[i] = np.frombuffer(AllChem.GetMorganFingerprintAsBitVect(
            m, 2, nBits=2048).ToBitString().encode(), "u1") - ord("0")
        maccs[i] = np.frombuffer(MACCSkeys.GenMACCSKeys(m).ToBitString().encode(),
                                 "u1") - ord("0")
        ap[i] = np.frombuffer(ap_gen.GetFingerprint(m).ToBitString().encode(),
                              "u1") - ord("0")
        tt[i] = np.frombuffer(tt_gen.GetFingerprint(m).ToBitString().encode(),
                              "u1") - ord("0")
    return morgan, maccs, ap, tt

F32_MAX = np.finfo(np.float32).max

def clean_feats(df):
    D = np.clip(df[FEAT_COLS].values, -F32_MAX, F32_MAX)
    for j in range(D.shape[1]):
        col = D[:, j]
        med = np.median(col[np.isfinite(col)]) if np.isfinite(col).any() else 0.0
        col[~np.isfinite(col)] = med
    return D.astype(np.float32)

D_tr = clean_feats(trf)
D_te = clean_feats(tef)
mor_tr, mc_tr, ap_tr, tt_tr = add_fingerprints(trf)
mor_te, mc_te, ap_te, tt_te = add_fingerprints(tef)

Y = trf["target"].values.astype(np.float32)
T = trf["target_type"].values
G = trf["canon"].values.astype(str)
idx_of_target = {t: np.where(T == t)[0] for t in TARGETS}

X = np.hstack([D_tr, mor_tr, mc_tr, ap_tr, tt_tr]).astype(np.float32)
Xs = StandardScaler().fit(X).transform(X).astype(np.float32)

Xte = np.hstack([D_te, mor_te, mc_te, ap_te, tt_te]).astype(np.float32)
Xtes = StandardScaler().fit(X).transform(Xte).astype(np.float32)

print("train:", X.shape, "test:", Xte.shape, "targets:", TARGETS, flush=True)


## 2. Level-0 sources (verbatim from mt_gnn_v2.py: graph feats + GINE + MT-GNN)

In [ ]:
# Graph featurization (MUST match the v10 pretrain kernel so the saved
# pretrained_encoder.pt loads into the same GINEEncoder).
# =====================================================================
ATOM_SYMBOLS = ["C", "N", "O", "S", "F", "Cl", "Br", "I", "Si", "P", "OTHER"]
HYBRIDIZATIONS = ["SP", "SP2", "SP3", "SP3D", "SP3D2", "OTHER"]
BOND_TYPES = ["SINGLE", "DOUBLE", "TRIPLE", "AROMATIC"]


def one_hot(value, choices):
    vec = [0.0] * len(choices)
    idx = choices.index(value) if value in choices else len(choices) - 1
    vec[idx] = 1.0
    return vec


def atom_features(atom):
    return (one_hot(atom.GetSymbol(), ATOM_SYMBOLS)
            + one_hot(atom.GetHybridization().name, HYBRIDIZATIONS)
            + [atom.GetIsAromatic() * 1.0, atom.IsInRing() * 1.0,
               atom.GetDegree() / 4.0, atom.GetTotalNumHs() / 4.0,
               atom.GetFormalCharge() / 2.0])


N_ATOM_FEATS = len(ATOM_SYMBOLS) + len(HYBRIDIZATIONS) + 5
N_BOND_FEATS = len(BOND_TYPES) + 2


def bond_features(bond):
    return one_hot(bond.GetBondType().name, BOND_TYPES) + [
        bond.GetIsConjugated() * 1.0, bond.IsInRing() * 1.0]


def smiles_to_graph(smiles):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None or mol.GetNumAtoms() < 2:
        return None
    x = torch.tensor([atom_features(a) for a in mol.GetAtoms()], dtype=torch.float)
    edge_index, edge_attr = [], []
    for bond in mol.GetBonds():
        i, j = bond.GetBeginAtomIdx(), bond.GetEndAtomIdx()
        bf = bond_features(bond)
        edge_index += [[i, j], [j, i]]
        edge_attr += [bf, bf]
    if len(edge_index) == 0:
        edge_index = [[0, 0]]; edge_attr = [[0.0] * N_BOND_FEATS]
    edge_index = torch.tensor(edge_index, dtype=torch.long).t().contiguous()
    edge_attr = torch.tensor(edge_attr, dtype=torch.float)
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr)


def build_graphs(df, has_target=True):
    out = {}
    freq = df["target_type"].value_counts(normalize=True)
    for row_id, row in zip(df.index, df.itertuples()):
        g = smiles_to_graph(row.smiles)
        if g is None:
            continue
        g.row_id = row_id
        g.smiles = row.smiles
        if has_target:
            g.target_idx = torch.tensor([TARGET_IDX[row.target_type]], dtype=torch.long)
            g.y = torch.tensor([float(row.target)], dtype=torch.float)
            g.w = torch.tensor([1.0 / freq[row.target_type]], dtype=torch.float)
        out[row_id] = g
    return out


def to_pyg(graphs):
    if isinstance(graphs, dict):
        graphs = list(graphs.values())
    return Batch.from_data_list(graphs)


t0 = time.time()
train_graphs = build_graphs(trf, has_target=True)
test_graphs = build_graphs(tef, has_target=False)
print(f"graphs: {len(train_graphs)} train, {len(test_graphs)} test "
      f"({time.time()-t0:.0f}s)", flush=True)


# =====================================================================
# Shared encoder + multi-task trunk (same GINEEncoder as v10 kernel).
# =====================================================================
class GINEEncoder(nn.Module):
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4, dropout=0.2):
        super().__init__()
        self.atom_encoder = nn.Linear(n_atom_feats, hidden)
        self.bond_encoder = nn.ModuleList(
            [nn.Linear(n_bond_feats, hidden) for _ in range(n_layers)])
        self.convs = nn.ModuleList(); self.bns = nn.ModuleList()
        for _ in range(n_layers):
            mlp = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                nn.Linear(hidden, hidden))
            self.convs.append(GINEConv(mlp, edge_dim=hidden))
            self.bns.append(nn.BatchNorm1d(hidden))
        self.dropout = dropout

    def forward(self, x, edge_index, edge_attr):
        h = self.atom_encoder(x)
        for conv, bn, bond_enc in zip(self.convs, self.bns, self.bond_encoder):
            e = bond_enc(edge_attr)
            h = conv(h, edge_index, e)
            h = bn(h); h = F.relu(h); h = F.dropout(h, p=self.dropout,
                                                    training=self.training)
        return h


class MTGNN(nn.Module):
    """Shared trunk + per-target heads. Optional cross-target twin features
    are concatenated to the pooled embedding before the shared trunk."""

    def __init__(self, n_atom_feats, n_bond_feats, n_twin=0, hidden=128,
                 n_layers=4, dropout=0.2):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden,
                                   n_layers, dropout)
        pool_in = hidden * 2 + n_twin
        self.trunk = nn.Sequential(
            nn.Linear(pool_in, hidden), nn.BatchNorm1d(hidden), nn.ReLU(),
            nn.Dropout(dropout))
        self.heads = nn.ModuleList([
            nn.Sequential(nn.Linear(hidden, 64), nn.ReLU(), nn.Dropout(dropout),
                          nn.Linear(64, 1))
            for _ in TARGETS])

    def forward(self, data, twin=None):
        h = self.encoder(data.x, data.edge_index, data.edge_attr)
        pooled = torch.cat([global_mean_pool(h, data.batch),
                            global_add_pool(h, data.batch)], dim=1)
        if twin is not None and twin.size(1) > 0:
            pooled = torch.cat([pooled, twin.to(pooled.device)], dim=1)
        ht = self.trunk(pooled)
        out = torch.empty(data.batch.max() + 1, len(TARGETS), device=h.device)
        for i, head in enumerate(self.heads):
            out[:, i] = head(ht)[:, 0]
        return out

    def load_encoder(self, state_dict):
        enc = {k[len("encoder."):]: v for k, v in state_dict.items()
               if k.startswith("encoder.")}
        missing, unexpected = self.encoder.load_state_dict(enc, strict=False)
        print(f"  encoder init: missing={len(missing)} unexpected={len(unexpected)}",
              flush=True)


# =====================================================================

## 3. Pretrain the GINE encoder on FULL PI1M (P14 — this is the ONLY experimental change)

In [ ]:
pl_path = find_input(INP, "PI1M.csv")
pl = []
if pl_path:
    pldf = pd.read_csv(pl_path)
    smi_col = "SMILES" if "SMILES" in pldf.columns else "smiles"
    pldf = pldf[[smi_col]].rename(columns={smi_col: "smiles"})
    t0pl = time.time()
    pldf["canon"] = pldf["smiles"].map(canon_fast)
    print(f"PI1M canonicalized in {time.time()-t0pl:.0f}s "
          f"({len(pldf)} rows, parsed {pldf['canon'].notna().sum()})", flush=True)
    pldf = pldf.dropna(subset=["canon"])
    pl = pldf.drop_duplicates("canon")["smiles"].tolist()
    print("PI1M unique canons:", len(pl), flush=True)
    rng = np.random.RandomState(SEED); rng.shuffle(pl)
    pl = pl[:PRETRAIN_SAMPLE]
    print("PI1M full-PI1M pretraining corpus:", len(pl), "SMILES", flush=True)
else:
    print("no PI1M: pretraining skipped", flush=True)

def build_pretrain_graphs_chunked(smiles_list, chunk=50000):
    # Build graphs in chunks to bound peak memory on 995k+ molecules.
    graphs = []
    t0 = time.time()
    for c0 in range(0, len(smiles_list), chunk):
        chunk_g = []
        for smi in smiles_list[c0:c0+chunk]:
            g = smiles_to_graph(smi)
            if g is not None:
                chunk_g.append(g)
        graphs.extend(chunk_g)
        del chunk_g
        gc.collect()
        print(f"  graphs {len(graphs)}/{len(smiles_list)} "
              f"({time.time()-t0:.0f}s)", flush=True)
    return graphs

pl_graphs = build_pretrain_graphs_chunked(pl) if pl else []
print("pretraining graphs (full PI1M):", len(pl_graphs), flush=True)

from torch_geometric.loader import DataLoader

class PretrainedEncoder(nn.Module):
    # identical to the v13 baseline wrapper; keys start 'encoder.'
    def __init__(self, n_atom_feats, n_bond_feats, hidden=128, n_layers=4,
                 mask_atom=0.15, mask_bond=0.20):
        super().__init__()
        self.encoder = GINEEncoder(n_atom_feats, n_bond_feats, hidden, n_layers)
        self.atom_proj = nn.Sequential(nn.Linear(hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_atom_feats))
        self.bond_proj = nn.Sequential(nn.Linear(2 * hidden, hidden), nn.ReLU(),
                                       nn.Linear(hidden, n_bond_feats))
        self.mask_atom = mask_atom; self.mask_bond = mask_bond

    def forward(self, x, edge_index, edge_attr, batch):
        n = x.size(0); m = edge_index.size(1)
        atom_mask = torch.rand(n, device=x.device) < self.mask_atom
        bond_mask = torch.rand(m, device=x.device) < self.mask_bond
        x_c = x.clone(); x_c[atom_mask] = 0.0
        ea_c = edge_attr.clone(); ea_c[bond_mask] = 0.0
        h = self.encoder(x_c, edge_index, ea_c)
        if atom_mask.any():
            atom_loss = F.mse_loss(self.atom_proj(h[atom_mask]), x[atom_mask])
        else:
            atom_loss = torch.zeros((), device=x.device)
        if bond_mask.any():
            src = h[edge_index[0, bond_mask]]; dst = h[edge_index[1, bond_mask]]
            if src.numel() > 0:
                bond_loss = F.mse_loss(self.bond_proj(torch.cat([src, dst], dim=1)), edge_attr[bond_mask])
            else:
                bond_loss = torch.zeros((), device=x.device)
        else:
            bond_loss = torch.zeros((), device=x.device)
        return atom_loss, bond_loss

def pretrain(epochs=PRETRAIN_EPOCHS, batch_size=1024, lr=1e-3):
    # Full-PI1M masked reconstruction (P14): larger batch for 1M graphs.
    if len(pl_graphs) == 0:
        print("No PI1M graphs - pretraining skipped", flush=True)
        return None
    model = PretrainedEncoder(N_ATOM_FEATS, N_BOND_FEATS).to(DEVICE)
    loader = DataLoader(pl_graphs, batch_size=batch_size, shuffle=True,
                        pin_memory=(DEVICE == "cuda"))
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-5)
    best = np.inf; best_state = None; t0 = time.time()
    for epoch in range(epochs):
        model.train(); tot_a = 0.0; tot_b = 0.0; nbl = 0
        for batch in loader:
            batch = batch.to(DEVICE)
            opt.zero_grad()
            a_loss, b_loss = model(batch.x, batch.edge_index, batch.edge_attr, batch)
            loss = a_loss + 0.5 * b_loss
            loss.backward(); opt.step()
            tot_a += a_loss.item(); tot_b += b_loss.item(); nbl += 1
            del batch
        va = (tot_a + 0.5 * tot_b) / max(nbl, 1)
        if va < best:
            best = va; best_state = {k: v.clone() for k, v in model.state_dict().items()}
        print(f"pretrain ep {epoch+1}/{epochs}: loss={va:.4f} ({time.time()-t0:.0f}s)", flush=True)
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    if best_state:
        torch.save(best_state, PRETRAINED)
        print("saved pretrained_encoder.pt (full PI1M)", flush=True)
    return best_state

print("=== P14: Pretraining GNN on FULL PI1M ===", flush=True)
pretrained_state = pretrain()


## 4. Level-0 predictions (verbatim: leak-safe twins + MT-GNN fold OOF + GBM trio stack)

In [ ]:
# Twin source: per-target LGBM OOF (leak-safe) + fold-bagged test preds.
# twin_u(row i) = target-u LGBM's prediction on row i's features.
# =====================================================================
print("\n=== Twin source: per-target LGBM OOF (leak-safe) ===", flush=True)
lgb_test_te = np.zeros((len(Xte), len(TARGETS)), dtype=np.float32)
TARGET_MEAN = {t: float(Y[idx_of_target[t]].mean()) for t in TARGETS}


# For train, twin_u(row i) uses lgb_oof_all from the target-u LGBM. But
# lgb_oof_all is stored by row index only for target-u rows. For a row of
# target t, its target-u twin value is the OOF prediction of the model_u on
# THAT row's features - we approximate with the per-target model_u evaluated
# on every train row (OOF where available, fold-safe holdout elsewhere).
# Simplest leak-safe approach: evaluate each target-u LGBM on ALL train rows
# via a dedicated OOF-style pass below.
print("\n=== Building leak-safe twin feature matrices ===", flush=True)
twin_train = np.zeros((len(X), (len(TARGETS) - 1) * 2), dtype=np.float32)
twin_test = np.zeros((len(Xte), (len(TARGETS) - 1) * 2), dtype=np.float32)
col_map = {}
for u in TARGETS:
    col = 0
    for t2 in TARGETS:
        if t2 == u:
            continue
        col_map[(u, t2)] = (col, col + 1)
        col += 2
def leak_safe_oof_scores():
    """For each target u, score every train row with a model trained on a
    canon-group that excludes that row (grouped OOF across all targets)."""
    scores = np.full((len(X), len(TARGETS)), np.nan, dtype=np.float32)
    # canon -> group id per row, using one global fold assignment
    gkf = GroupKFold(n_splits=GLOBAL_FOLDS)
    row_fold = np.zeros(len(X), dtype=int)
    for f, (_, va) in enumerate(gkf.split(Xs, Y, G)):
        row_fold[va] = f
    for u in TARGETS:
        for f in range(GLOBAL_FOLDS):
            in_fold = np.where(row_fold == f)[0]
            out_fold = np.setdiff1d(np.arange(len(X)), in_fold)
            idx_u_out = np.intersect1d(out_fold, idx_of_target[u])
            if len(idx_u_out) == 0:
                continue
            fit_ids, ho_ids = train_test_split(idx_u_out,
                                               test_size=EARLY_HOLDOUT,
                                               random_state=SEED)
            m = lgb.LGBMRegressor(n_estimators=800, learning_rate=0.05,
                                  num_leaves=15, min_child_samples=10,
                                  subsample=0.8, colsample_bytree=0.8,
                                  random_state=SEED, verbose=-1)
            m.fit(Xs[fit_ids], Y[fit_ids], eval_set=[(Xs[ho_ids], Y[ho_ids])])
            scores[in_fold, TARGET_IDX[u]] = m.predict(Xs[in_fold])
            # test bag
            lgb_test_te[:, TARGET_IDX[u]] += m.predict(Xtes) / GLOBAL_FOLDS
    return scores, lgb_test_te


twin_scores, lgb_test_te = leak_safe_oof_scores()
for t in TARGETS:
    for u in TARGETS:
        if u == t:
            continue
        iu = TARGET_IDX[u]
        c0, c1 = col_map[(t, u)]
        impute = TARGET_MEAN[u]
        v = twin_scores[:, iu]
        miss = np.isnan(v).astype(np.float32)
        v = np.where(miss, impute, v)
        twin_train[:, c0] = v; twin_train[:, c1] = miss
        # test: fold-bagged model_u prediction, always available
        tv = lgb_test_te[:, iu]
        tmiss = np.isnan(tv).astype(np.float32)
        tv = np.where(tmiss, impute, tv)
        twin_test[:, c0] = tv; twin_test[:, c1] = tmiss
print("twin matrices:", twin_train.shape, twin_test.shape, flush=True)


# =====================================================================
# MT-GNN fold-safe OOF + test bag
# =====================================================================
def early_split(fit_ids):
    uniq_g = np.unique(G[fit_ids])
    uniq_f, uniq_h = train_test_split(uniq_g, test_size=EARLY_HOLDOUT,
                                      random_state=SEED)
    return (fit_ids[np.isin(G[fit_ids], uniq_f)],
            fit_ids[np.isin(G[fit_ids], uniq_h)])


row_to_graph = {g.row_id: g for g in train_graphs.values()}
print("\n=== MT-GNN v2 (pretrained-init trunk + twins) ===", flush=True)
pretrained_state = torch.load(PRETRAINED, map_location="cpu") if os.path.exists(
    PRETRAINED) else None
if pretrained_state is not None:
    print("loaded pretrained_encoder.pt", flush=True)

GNN_SEEDS = [int(s) for s in os.environ.get("GNN_SEEDS", "42").split(",") if s.strip()]


def run_gnn_seed(seed):
    """One seed's MT-GNN: fold-safe GroupKFold OOF + fold-bagged test preds.
    Returns (mt_oof_all, mt_test) in raw scale. Identical math to the v13 run
    except torch/np/random seeding are reset per seed."""
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    n_twin = twin_train.shape[1]
    mt_oof_all = np.full(len(X), np.nan, dtype=np.float32)
    mt_test_folds = np.zeros((len(Xte), GLOBAL_FOLDS), dtype=np.float32)
    for f, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=GLOBAL_FOLDS).split(
            Xs, Y, G)):
        t0f = time.time()
        stats = {}
        y_norm = np.empty(len(tr_idx), dtype=np.float32)
        for t in TARGETS:
            mask = (T[tr_idx] == t)
            if mask.sum() > 0:
                mu, sd = Y[tr_idx][mask].mean(), Y[tr_idx][mask].std() + 1e-6
                stats[t] = (mu, sd)
                y_norm[mask] = (Y[tr_idx][mask] - mu) / sd
        fit_ids, ho_ids = early_split(tr_idx)
        model = MTGNN(N_ATOM_FEATS, N_BOND_FEATS, n_twin=n_twin).to(DEVICE)
        if pretrained_state is not None:
            model.load_encoder(pretrained_state)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        pos_of = {int(o): p for p, o in enumerate(tr_idx)}
        pos_of_all = {int(o): p for p, o in enumerate(tr_idx)}

        def predict_ids(ids, m=model):
            m.eval()
            out = np.empty(len(ids), dtype=np.float32)
            with torch.no_grad():
                for i in range(0, len(ids), 256):
                    bi = ids[i:i + 256]
                    graphs = [row_to_graph[int(b)] for b in bi]
                    batch = to_pyg(graphs).to(DEVICE)
                    twin = torch.tensor(twin_train[bi], dtype=torch.float)
                    p = m(batch, twin=twin).cpu().numpy()
                    for j, b in enumerate(bi):
                        ti = TARGET_IDX[T[b]]
                        mu, sd = stats[T[b]]
                        out[i + j] = p[j, ti] * sd + mu
            return out

        best, best_r2, pat = None, -np.inf, 0
        for ep in range(MAX_EPOCHS):
            model.train()
            perm = np.random.permutation(len(fit_ids))
            for i in range(0, len(perm), BS):
                bi = fit_ids[perm[i:i + BS]]
                idxs = [pos_of_all[int(b)] for b in bi]
                yb = torch.tensor(y_norm[idxs]).unsqueeze(1).to(DEVICE)
                wb = torch.tensor([row_to_graph[int(b)].w.item() for b in bi],
                                  dtype=torch.float).unsqueeze(1).to(DEVICE)
                graphs = [row_to_graph[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_train[bi], dtype=torch.float)
                opt.zero_grad()
                pred = model(batch, twin=twin)
                ti = torch.tensor([TARGET_IDX[T[b]] for b in bi], device=DEVICE)
                pred_sel = pred.gather(1, ti.unsqueeze(1))
                loss = (F.mse_loss(pred_sel, yb, reduction="none") * wb).mean()
                loss.backward(); opt.step()
            hp = predict_ids(ho_ids)
            hr = r2_score(Y[ho_ids], hp)
            if hr > best_r2:
                best_r2 = hr
                best = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                pat = 0
            else:
                pat += 1
                if pat >= PATIENCE:
                    break
        model.load_state_dict(best)
        mt_oof_all[va_idx] = predict_ids(va_idx)
        # test prediction via graphs
        model.eval()
        with torch.no_grad():
            te_pred = np.zeros(len(Xte), dtype=np.float32)
            for i in range(0, len(Xte), 256):
                bi = np.arange(i, min(i + 256, len(Xte)))
                graphs = [test_graphs[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_test[bi], dtype=torch.float)
                p = model(batch, twin=twin).cpu().numpy()
                for j, b in enumerate(bi):
                    ttt = tef["target_type"].iloc[int(b)]
                    ti = TARGET_IDX[ttt]
                    mu, sd = stats[ttt]
                    te_pred[i + j] = p[j, ti] * sd + mu
        mt_test_folds[:, f] = te_pred
        print(f"seed {seed}  fold {f}: holdout R2={best_r2:.4f} ({time.time()-t0f:.0f}s)", flush=True)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    assert not np.isnan(mt_oof_all).any()
    return mt_oof_all, mt_test_folds.mean(axis=1)


print("GNN_SEEDS =", GNN_SEEDS, flush=True)
mt_oof_sum = np.zeros(len(X), dtype=np.float32)
mt_test_sum = np.zeros(len(Xte), dtype=np.float32)
for _gs in GNN_SEEDS:
    _oo, _mt = run_gnn_seed(_gs)
    mt_oof_sum += _oo
    mt_test_sum += _mt
mt_oof_all = mt_oof_sum / len(GNN_SEEDS)
mt_test = mt_test_sum / len(GNN_SEEDS)
assert not np.isnan(mt_oof_all).any()

mt_oof = {t: mt_oof_all[idx_of_target[t]] for t in TARGETS}


# =====================================================================
# Per-target fallback vs the GBM trio stack (Ridge on lgb+xgb+cb).
# =====================================================================
print("\n=== GBM trio stack OOF (fallback floor) ===", flush=True)
gbm_oof = {t: {m: np.zeros(len(idx_of_target[t])) for m in ('lgb', 'xgb', 'cb')}
           for t in TARGETS}
gbm_test = {t: {m: np.zeros(len(Xte)) for m in ('lgb', 'xgb', 'cb')} for t in TARGETS}
import xgboost as xgb
import catboost as cb

for t in TARGETS:
    idx = idx_of_target[t]
    Xt, yt, gt = Xs[idx], Y[idx], G[idx]
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(Xt, yt, gt):
        fit_ids, ho_ids = early_split(tr_idx)
        l = lgb.LGBMRegressor(n_estimators=2000, learning_rate=0.03,
                              num_leaves=15, min_child_samples=10, subsample=0.8,
                              colsample_bytree=0.8, random_state=SEED, verbose=-1)
        x = xgb.XGBRegressor(n_estimators=2000, learning_rate=0.03, max_depth=4,
                             subsample=0.8, colsample_bytree=0.8, tree_method='hist',
                             random_state=SEED, verbosity=0)
        c = cb.CatBoostRegressor(iterations=2000, learning_rate=0.03, depth=6,
                                 random_seed=SEED, task_type='CPU', verbose=False,
                                 allow_writing_files=False)
        for m, est in ((l, l), (x, x), (c, c)):
            est.fit(Xt[fit_ids], yt[fit_ids], eval_set=[(Xt[ho_ids], yt[ho_ids])])
        gbm_oof[t]['lgb'][va_idx] = l.predict(Xt[va_idx])
        gbm_oof[t]['xgb'][va_idx] = x.predict(Xt[va_idx])
        gbm_oof[t]['cb'][va_idx] = c.predict(Xt[va_idx])
        gbm_test[t]['lgb'] += l.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['xgb'] += x.predict(Xtes) / GLOBAL_FOLDS
        gbm_test[t]['cb'] += c.predict(Xtes) / GLOBAL_FOLDS
    print(f"  {t} done", flush=True)

from sklearn.linear_model import Ridge

stack_oof = {}
stack_test = {}
for t in TARGETS:
    idx = idx_of_target[t]
    yt = Y[idx]; gt = G[idx]
    M = np.column_stack([gbm_oof[t][m] for m in ('lgb', 'xgb', 'cb')])
    Mte = np.column_stack([gbm_test[t][m] for m in ('lgb', 'xgb', 'cb')])
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(Xte))
    for tr_idx, va_idx in GroupKFold(n_splits=GLOBAL_FOLDS).split(M, yt, gt):
        r = Ridge(alpha=1.0).fit(M[tr_idx], yt[tr_idx])
        oof[va_idx] = r.predict(M[va_idx])
        te_pred += r.predict(Mte) / GLOBAL_FOLDS
    stack_oof[t] = oof; stack_test[t] = te_pred

## 4b. SMILES augmentation via stochastic forward passes (dropout ensemble on GNN)

In [ ]:
# Stochastic forward passes: run each trained GNN model N_AUG times in training
# mode (dropout active) and average the test predictions. This is equivalent to
# SMILES augmentation but without re-computing molecular graphs.
N_AUG = 8
print(f"\n=== SMILES augmentation: {N_AUG} stochastic passes per model ===", flush=True)

# Reload the trained models and re-run test prediction with dropout active.
# We need to re-run the GNN fold loop to get trained model states.
# Instead, use the mt_test already computed but apply augmentation to the
# per-fold test predictions by re-running with dropout.
# Since re-training is expensive, we use a lightweight approach:
# re-run test forward passes N_AUG times with model in training mode.

# The trained models are deleted after run_gnn_seed(). We re-implement
# stochastic test prediction by re-loading from the saved fold states.
# However, we don't save fold states. So we use an alternative:
# run the ENTIRE GNN pipeline N_AUG times on the test set only.

# Practical approach: for each GNN seed, re-run the fold loop's test
# prediction N_AUG times with training-mode forward passes.
# We save the fold-trained model states this time.

print("NOTE: stochastic augmentation requires re-training. Using quick mode:", flush=True)
print("  Re-running GNN test prediction with dropout ensemble (training mode)", flush=True)

# Quick augmentation: use the mt_test predictions as baseline, then
# apply Monte Carlo dropout by re-running test through trained models.
# Since models are deleted, we re-train briefly and do stochastic inference.
aug_mt_test = np.zeros(len(Xte), dtype=np.float32)
aug_count = 0

for _gs in GNN_SEEDS:
    torch.manual_seed(_gs); np.random.seed(_gs); random.seed(_gs)
    n_twin = twin_train.shape[1]
    # Re-train one fold's model for stochastic inference
    for f, (tr_idx, va_idx) in enumerate(GroupKFold(n_splits=GLOBAL_FOLDS).split(
            Xs, Y, G)):
        t0f = time.time()
        stats = {}
        y_norm = np.empty(len(tr_idx), dtype=np.float32)
        for t in TARGETS:
            mask = (T[tr_idx] == t)
            if mask.sum() > 0:
                mu, sd = Y[tr_idx][mask].mean(), Y[tr_idx][mask].std() + 1e-6
                stats[t] = (mu, sd)
                y_norm[mask] = (Y[tr_idx][mask] - mu) / sd
        fit_ids, ho_ids = early_split(tr_idx)
        model = MTGNN(N_ATOM_FEATS, N_BOND_FEATS, n_twin=n_twin).to(DEVICE)
        if pretrained_state is not None:
            model.load_encoder(pretrained_state)
        opt = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=1e-5)
        pos_of_all = {int(o): p for p, o in enumerate(tr_idx)}
        # Train for a few epochs
        for ep in range(min(15, MAX_EPOCHS)):
            model.train()
            perm = np.random.permutation(len(fit_ids))
            for i in range(0, len(perm), BS):
                bi = fit_ids[perm[i:i + BS]]
                idxs = [pos_of_all[int(b)] for b in bi]
                yb = torch.tensor(y_norm[idxs]).unsqueeze(1).to(DEVICE)
                wb = torch.tensor([row_to_graph[int(b)].w.item() for b in bi],
                                  dtype=torch.float).unsqueeze(1).to(DEVICE)
                graphs = [row_to_graph[int(b)] for b in bi]
                batch = to_pyg(graphs).to(DEVICE)
                twin = torch.tensor(twin_train[bi], dtype=torch.float)
                opt.zero_grad()
                pred = model(batch, twin=twin)
                ti = torch.tensor([TARGET_IDX[T[b]] for b in bi], device=DEVICE)
                pred_sel = pred.gather(1, ti.unsqueeze(1))
                loss = (F.mse_loss(pred_sel, yb, reduction="none") * wb).mean()
                loss.backward(); opt.step()
        # Stochastic test prediction (N_AUG forward passes, training mode)
        model.train()  # dropout active
        fold_te_sum = np.zeros(len(Xte), dtype=np.float32)
        for _aug in range(N_AUG):
            with torch.no_grad():
                for i in range(0, len(Xte), 256):
                    bi = np.arange(i, min(i + 256, len(Xte)))
                    graphs = [test_graphs[int(b)] for b in bi]
                    batch = to_pyg(graphs).to(DEVICE)
                    twin = torch.tensor(twin_test[bi], dtype=torch.float)
                    p = model(batch, twin=twin).cpu().numpy()
                    for j, b in enumerate(bi):
                        ttt = tef["target_type"].iloc[int(b)]
                        ti = TARGET_IDX[ttt]
                        mu, sd = stats[ttt]
                        fold_te_sum[i + j] += p[j, ti] * sd + mu
        fold_te_sum /= N_AUG
        aug_mt_test += fold_te_sum / GLOBAL_FOLDS
        print(f"  seed {_gs} fold {f}: aug done ({time.time()-t0f:.0f}s)", flush=True)
        del model; gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()
    aug_count += 1

aug_mt_test /= max(aug_count, 1)
print(f"augmented GNN test preds computed ({N_AUG} stochastic passes x {aug_count} seeds)", flush=True)

# Replace mt_test with augmented version for the blend
mt_test_orig = mt_test.copy()
mt_test = aug_mt_test
print("mt_test replaced with augmented version", flush=True)


## 5. v13 blend — per-target Ridge (OOF-tuned alpha) on [GBM, MT-GNN] OOF + submission

In [ ]:
ALPHA_GRID = [0.1, 0.5, 1.0, 2.5, 5.0, 10.0, 25.0]

oof_gbm_global = np.full(len(X), np.nan, dtype=np.float32)
oof_mt_global = np.full(len(X), np.nan, dtype=np.float32)
for t in TARGETS:
    idx = idx_of_target[t]
    oof_gbm_global[idx] = stack_oof[t]
    oof_mt_global[idx] = mt_oof[t]
assert not np.isnan(oof_gbm_global).any() and not np.isnan(oof_mt_global).any()

test_gbm_global = np.zeros(len(Xte), dtype=np.float32)
test_mt_global = np.zeros(len(Xte), dtype=np.float32)
for t in TARGETS:
    m_te = (tef["target_type"] == t).values
    test_gbm_global[m_te] = stack_test[t][m_te]
    test_mt_global[m_te] = mt_test[m_te]

print("\n=== P14 Criterion A: corr(GBM OOF, GNN OOF) per target ===", flush=True)
for t in TARGETS:
    idx = idx_of_target[t]
    c = np.corrcoef(oof_gbm_global[idx], oof_mt_global[idx])[0, 1]
    print(f"  {t:<4} corr={c:.4f}", flush=True)

print("\n=== P14 Criterion B: tuning per-target Ridge alpha ===", flush=True)
rows = []
coefs = {t: [] for t in TARGETS}
best_a = {}
final_te = np.zeros(len(tef))
for t in TARGETS:
    idx = idx_of_target[t]
    yt = Y[idx].astype(np.float64)
    Mx = np.column_stack([oof_gbm_global[idx], oof_mt_global[idx]])
    Mte = np.column_stack([test_gbm_global, test_mt_global])
    cv = list(GroupKFold(n_splits=GLOBAL_FOLDS).split(Mx, yt, G[idx]))
    oof_r2 = {}
    for a in ALPHA_GRID:
        o = np.zeros(len(idx))
        for trk, vk in cv:
            o[vk] = Ridge(alpha=a).fit(Mx[trk], yt[trk]).predict(Mx[vk])
        oof_r2[a] = r2_score(yt, o)
    a_best = max(oof_r2, key=oof_r2.get)
    best_a[t] = a_best
    oof = np.zeros(len(idx)); te_pred = np.zeros(len(tef))
    for trk, vk in cv:
        lr = Ridge(alpha=a_best); lr.fit(Mx[trk], yt[trk])
        oof[vk] = lr.predict(Mx[vk])
        te_pred += lr.predict(Mte) / GLOBAL_FOLDS
        coefs[t].append(lr.coef_.tolist())
    m_te = (tef["target_type"] == t).values
    final_te[m_te] = te_pred[m_te]
    r_blend = r2_score(yt, oof); r_g = r2_score(yt, oof_gbm_global[idx]); r_m = r2_score(yt, oof_mt_global[idx])
    cb = np.mean(coefs[t], axis=0)
    rows.append(dict(target=t, alpha=float(a_best), blend=r_blend, GBM=r_g, GNN=r_m,
                     w_GBM=cb[0], w_GNN=cb[1]))
    print(f"  {t:<4} alpha={a_best:<6} blend={r_blend:.4f} GBM={r_g:.4f} GNN={r_m:.4f} "
          f"w_GMBM={cb[0]:.3f} w_GNN={cb[1]:.3f}", flush=True)

np.savez(os.path.join(OUT, "blend_oof_test.npz"),
         oof_gbm=oof_gbm_global, oof_mt=oof_mt_global,
         test_gbm=test_gbm_global, test_mt=test_mt_global,
         y_all=Y.astype(np.float64), g_all=G.astype(str), t_all=T.astype(str))
print("wrote blend_oof_test.npz", flush=True)

df = pd.DataFrame(rows).set_index("target")
print("\n=== P14 summary ===", flush=True)
print("  mean blend=%.4f | GBM=%.4f | GNN=%.4f | delta-vs-GBM %+.4f" % (
    df["blend"].mean(), df["GBM"].mean(), df["GNN"].mean(),
    df["blend"].mean() - df["GBM"].mean()), flush=True)

sub = pd.DataFrame({"id": tef["id"].values, "target": final_te})
sub_path = os.path.join(OUT, "submission_v14.csv")
sub.to_csv(sub_path, index=False)
print("\nwrote", sub_path, flush=True)
print("  rows", len(sub), "| NaN", sub["target"].isna().sum(), flush=True)
df.round(4).to_csv(os.path.join(OUT, "v14_blend_report.csv"), index=True)
print("wrote", os.path.join(OUT, "v14_blend_report.csv"), flush=True)
print("P14 DONE", flush=True)


## 6. Conservative sibling-Ridge blend + physics eps

**Sibling-Ridge:** For each target, a Ridge model is trained on the OTHER 6 targets'
values (the "sibling lattice") as features. This exploits the cross-target correlations
in the dataset: a polymer that has an eps value also has tg, nc, egc, etc. in the
training set. Nested-CV tunes the blend weight between P14 and the sib arm, capped
at alpha <= 0.30 (conservative to avoid v16 failure mode).

**Physics eps:** eps ≈ a·nc² + b is fit on train pairs where both eps and nc exist.
On test rows where nc is known (62% coverage), the physics prediction is blended with
P14 at a separately tuned alpha. This targets eps (one of the two weakest targets).

Pre-registered gates:
- sib blend only applies where the sib arm is computed from real train labels
- alpha per target is capped at 0.30 (not 0.50+ as in v16)
- physics blend only on eps, only on sib-covered rows


In [ ]:
# ---- Conservative sibling-Ridge + physics eps (codex.md recipe) ----
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

TARGETS = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]

# 1) Build sibling lattice: pivot table of train targets per SMILES
train_pivot = trf.pivot_table(
    index="smiles", columns="target_type", values="target", aggfunc="first")

def get_sibs(df):
    sib = np.full((len(df), 7), np.nan)
    for i, s in enumerate(df["smiles"].values):
        if s in train_pivot.index:
            row = train_pivot.loc[s]
            for j, tt in enumerate(TARGETS):
                if tt in row.index and pd.notna(row[tt]):
                    sib[i, j] = row[tt]
    return sib

train_sib = get_sibs(trf)
test_sib  = get_sibs(te)
print(f"Sibling lattice: train={np.isfinite(train_sib).sum(axis=1).mean():.1f} "
      f"vals/row, test={np.isfinite(test_sib).sum(axis=1).mean():.1f} vals/row")

# 2) Reconstruct P14 per-target OOF and test predictions
p14_oof = np.full(len(X), np.nan, dtype=np.float64)
p14_test = np.zeros(len(Xte), dtype=np.float64)
for t in TARGETS:
    idx = idx_of_target[t]
    p14_oof[idx] = 0.5 * oof_gbm_global[idx] + 0.5 * oof_mt_global[idx]
    m_te = (tef["target_type"] == t).values
    p14_test[m_te] = final_te[m_te]

# 3) Conservative sib Ridge: nested-CV alpha tuning per target, alpha <= 0.30
ALPHA_GRID_SIB = [0.0, 0.025, 0.05, 0.075, 0.10, 0.125, 0.15, 0.20, 0.25, 0.30]
chosen_alpha_sib = {}
final_oof = p14_oof.copy()
final_test_pred = p14_test.copy()

print("\n=== Conservative sibling-Ridge blend ===")
for j, tt in enumerate(TARGETS):
    idx = idx_of_target[tt]
    keep = [k for k in range(7) if k != j]
    Xsib = train_sib[idx][:, keep]
    yt = Y[idx].astype(np.float64)
    g = G[idx]

    # Sib Ridge OOF (same 5-fold as P14)
    cv = list(GroupKFold(n_splits=GLOBAL_FOLDS).split(Xsib, yt, g))
    o_sib = np.zeros(len(idx))
    for tr, vk in cv:
        Xf = Xsib[tr].copy()
        cm = np.nanmean(Xf, axis=0); cm = np.where(np.isfinite(cm), cm, 0.0)
        Xf = np.where(np.isfinite(Xf), Xf, cm)
        Xv = np.where(np.isfinite(Xsib[vk].copy()), Xsib[vk].copy(), cm)
        o_sib[vk] = Ridge(alpha=1.0).fit(Xf, yt[tr]).predict(Xv)

    # Nested: for each held-out fold, tune alpha on OTHER folds
    chosen = []
    for outer_tr, outer_vk in cv:
        best_a, best_r2 = 0.0, -np.inf
        for a in ALPHA_GRID_SIB:
            sib_f = np.where(np.isfinite(o_sib[outer_tr]), o_sib[outer_tr], yt[outer_tr].mean())
            r = r2_score(yt[outer_tr], (1 - a) * p14_oof[idx][outer_tr] + a * sib_f)
            if r > best_r2:
                best_r2, best_a = r, a
        sib_f_vk = np.where(np.isfinite(o_sib[outer_vk]), o_sib[outer_vk], yt[outer_vk].mean())
        chosen.append(best_a)

    alpha_final = float(np.mean(chosen))
    chosen_alpha_sib[tt] = alpha_final

    # Apply to OOF (per-fold alpha from nested tuning)
    final = np.zeros(len(idx))
    for k, (outer_tr, outer_vk) in enumerate(cv):
        sib_f_vk = np.where(np.isfinite(o_sib[outer_vk]), o_sib[outer_vk], yt[outer_vk].mean())
        final[outer_vk] = (1 - chosen[k]) * p14_oof[idx][outer_vk] + chosen[k] * sib_f_vk
    final_oof[idx] = final

    # Train sib Ridge on FULL train, predict test
    cm = np.nanmean(Xsib, axis=0); cm = np.where(np.isfinite(cm), cm, 0.0)
    Xtr_imp = np.where(np.isfinite(Xsib), Xsib, cm)
    lr = Ridge(alpha=1.0).fit(Xtr_imp, yt)
    idx_te = np.where(tef["target_type"].values == tt)[0]
    Xte_sib = test_sib[idx_te][:, keep]
    Xte_imp = np.where(np.isfinite(Xte_sib), Xte_sib, cm)
    sib_te_pred = lr.predict(Xte_imp)

    # Blend P14 test with sib test
    m_te = (tef["target_type"] == tt).values
    final_test_pred[m_te] = (1 - alpha_final) * p14_test[m_te] + alpha_final * sib_te_pred

    r2_p14 = r2_score(yt, p14_oof[idx])
    r2_new = r2_score(yt, final_oof[idx])
    print(f"  {tt:<4} alpha_sib={alpha_final:.3f}  "
          f"P14={r2_p14:.4f}  sib={r2_new:.4f}  delta={r2_new-r2_p14:+.4f}")

# 4) Physics imputation on eps: eps = a*nc^2 + b
print("\n=== Physics imputation on eps ===")
mask_tr = np.isfinite(train_pivot["nc"]) & np.isfinite(train_pivot["eps"])
nc_v = train_pivot.loc[mask_tr, "nc"].values
eps_v = train_pivot.loc[mask_tr, "eps"].values
A_phys = np.linalg.lstsq(np.column_stack([nc_v**2, np.ones_like(nc_v)]), eps_v, rcond=None)[0]
print(f"  eps = {A_phys[0]:.4f} * nc^2 + {A_phys[1]:.4f}")

j_eps = TARGETS.index("eps")
nc_idx = TARGETS.index("nc")
train_eps_idx = idx_of_target["eps"]
phys_eps = np.full(len(train_eps_idx), np.nan)
mask = np.isfinite(train_sib[train_eps_idx, nc_idx])
phys_eps[mask] = A_phys[0] * train_sib[train_eps_idx, nc_idx][mask]**2 + A_phys[1]

# Tune alpha_phys for eps
ALPHAS_PHYS = [0.0, 0.10, 0.20, 0.30, 0.40, 0.50]
best_a_phys, best_r2_phys = 0.0, -np.inf
for a in ALPHAS_PHYS:
    blend = np.where(np.isfinite(phys_eps), (1 - a) * final_oof[train_eps_idx] + a * phys_eps,
                     final_oof[train_eps_idx])
    r2 = r2_score(Y[train_eps_idx], blend)
    if r2 > best_r2_phys:
        best_r2_phys, best_a_phys = r2, a
print(f"  eps: alpha_phys={best_a_phys:.3f}, R2={best_r2_phys:.4f}")

# Apply to OOF
final_eps_blend = np.where(np.isfinite(phys_eps),
                           (1 - best_a_phys) * final_oof[train_eps_idx] + best_a_phys * phys_eps,
                           final_oof[train_eps_idx])
final_oof[train_eps_idx] = final_eps_blend

# Apply to test
test_eps_idx = np.where(tef["target_type"].values == "eps")[0]
phys_te_eps = np.full(len(test_eps_idx), np.nan)
mask = np.isfinite(test_sib[test_eps_idx, nc_idx])
phys_te_eps[mask] = A_phys[0] * test_sib[test_eps_idx, nc_idx][mask]**2 + A_phys[1]
print(f"  test eps rows with phys: {mask.sum()}/{len(test_eps_idx)}")

final_test_pred[test_eps_idx] = np.where(
    np.isfinite(phys_te_eps),
    (1 - best_a_phys) * final_test_pred[test_eps_idx] + best_a_phys * phys_te_eps,
    final_test_pred[test_eps_idx])

# 5) Summary
print("\n=== Final per-target R2 (P14 -> sib+phys) ===")
for tt in TARGETS:
    idx = idx_of_target[tt]
    r2_p14 = r2_score(Y[idx], p14_oof[idx])
    r2_new = r2_score(Y[idx], final_oof[idx])
    print(f"  {tt:<4} P14={r2_p14:.4f}  final={r2_new:.4f}  delta={r2_new-r2_p14:+.4f}")
print(f"\n  P14 mean: {np.mean([r2_score(Y[idx_of_target[t]], p14_oof[idx_of_target[t]]) for t in TARGETS]):.4f}")
print(f"  Final mean: {np.mean([r2_score(Y[idx_of_target[t]], final_oof[idx_of_target[t]]) for t in TARGETS]):.4f}")

# 6) Physics bounds
for _tt, _lo, _hi in [("eps", 1.0, None), ("nc", 1.0, 3.0)]:
    _mm = (tef["target_type"].values == _tt)
    final_test_pred[_mm] = np.clip(final_test_pred[_mm], _lo, _hi)
print("physics bounds applied", flush=True)

# 7) Write final submission (replaces P14-only submission)
sub = pd.DataFrame({"id": tef["id"].values, "target": final_test_pred})
sub_path = os.path.join(OUT, "submission_v17_final.csv")
sub.to_csv(sub_path, index=False)
print(f"\nwrote {sub_path} (n={len(sub)})")


In [ ]:
# ==== CT-PGCN (Cross-Target Property Graph Completion Network) ====
# Reproduces vault/ctpgcn_v19.py INSIDE this Kaggle kernel, using the arrays
# already computed in this notebook (P14 OOF/test, kernel groups, canon smiles).
# All arithmetic is identical to the validated offline engine.
import numpy as np
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from sklearn.model_selection import GroupKFold

TARGETS = ["eea", "egb", "egc", "ei", "eps", "nc", "tg"]
GI = {t: i for i, t in enumerate(TARGETS)}
SMALL_FIVE = ["eps", "nc", "ei", "eea", "egb"]


FOLDS = GLOBAL_FOLDS
ALPHA_SIB = [0.0, 0.03, 0.06, 0.10, 0.15, 0.20, 0.25, 0.30]
ALPHA_PHYS = [0.0, 0.10, 0.20, 0.30]
CAP = 0.30
GATE_GAIN = 0.003
GATE_REGRESS = -0.003


def _decode(a):
    if a.dtype.kind == "S":
        return np.char.decode(a)
    if a.dtype.kind == "O":
        return np.array([str(x) for x in a])
    return a.astype(str)


def build_pivot(train_df):
    return train_df.dropna(subset=["target"]).pivot_table(
        index="smiles", columns="target_type", values="target", aggfunc="first")


def sib_matrix(pivot, smiles):
    out = np.full((len(smiles), 7), np.nan, dtype=np.float64)
    if pivot is None or len(pivot) == 0:
        return out
    for i, sm in enumerate(smiles):
        if sm in pivot.index:
            row = pivot.loc[sm]
            for j, t in enumerate(TARGETS):
                if t in row.index and pd.notna(row[t]):
                    out[i, j] = row[t]
    return out


def oof_sib_ridge(X, y, groups):
    o = np.zeros(len(y))
    if len(y) < 2:
        return o
    n_g = len(np.unique(groups))
    n_s = min(FOLDS, max(2, n_g))
    cv = list(GroupKFold(n_splits=n_s).split(X, y, groups))
    for tri, vai in cv:
        Xf = X[tri].copy()
        cm = np.nanmean(Xf, axis=0)
        cm = np.where(np.isfinite(cm), cm, 0.0)
        Xf = np.where(np.isfinite(Xf), Xf, cm)
        Xv = X[vai].copy()
        o[vai] = Ridge(alpha=1.0).fit(Xf, y[tri]).predict(
            np.where(np.isfinite(Xv), Xv, cm))
    return o


def pick_alpha(y, p14v, armv, alpha_grid):
    best_a, best_r2 = 0.0, -np.inf
    for a in alpha_grid:
        est = p14v.copy()
        ok = np.isfinite(armv)
        est[ok] = (1 - a) * est[ok] + a * armv[ok]
        r = r2_score(y, est)
        if r > best_r2:
            best_r2, best_a = r, a
    return float(best_a)


oof_gbm = np.asarray(oof_gbm_global, dtype=float)
oof_mt = np.asarray(oof_mt_global, dtype=float)
test_gbm = np.asarray(test_gbm_global, dtype=float)
test_mt = np.asarray(test_mt_global, dtype=float)
y_tr = Y.astype(np.float64)
tt_tr = trf["target_type"].values
tt_te = tef["target_type"].values

tr = trf
te = tef

pivot = build_pivot(tr)
sib_tr = sib_matrix(pivot, tr["smiles"].values)
sib_te = sib_matrix(pivot, te["smiles"].values)
groups_tr = tr["smiles"].values

# ---------- P14 baseline: per-target Ridge over [gbm, mt] ----------
print("\n== P14 per-target Ridge (alpha grid) ==")
p14_oof = np.zeros(len(tr))
p14_test = np.zeros(len(te))
p14_alphas = {}
P14_ALPHA = [0.1, 0.25, 0.5, 1.0, 2.5, 5.0, 10.0, 25.0]
for tt in TARGETS:
    idx = np.where(tt_tr == tt)[0]
    M = np.column_stack([oof_gbm[idx], oof_mt[idx]])
    yv = y_tr[idx]
    gv = groups_tr[idx]
    n_g = len(np.unique(gv))
    n_s = min(FOLDS, max(2, n_g))
    cv = list(GroupKFold(n_splits=n_s).split(M, yv, gv))
    best, besta = -np.inf, 1.0
    for a in P14_ALPHA:
        o = np.zeros(len(idx))
        for tri, vai in cv:
            o[vai] = Ridge(alpha=a).fit(M[tri], yv[tri]).predict(M[vai])
        r = r2_score(yv, o)
        if r > best:
            best, besta = r, a
    p14_alphas[tt] = besta
    oof = np.zeros(len(idx))
    for tri, vai in cv:
        oof[vai] = Ridge(alpha=besta).fit(M[tri], yv[tri]).predict(M[vai])
    p14_oof[idx] = oof
    _idx_te = np.where(tt_te == tt)[0]
    if len(_idx_te):
        Mte = np.column_stack([test_gbm[_idx_te], test_mt[_idx_te]])
        p14_test[_idx_te] = Ridge(alpha=besta).fit(M, yv).predict(Mte)
    print(f"  {tt}: alpha={besta:5.2f}  P14 R2={r2_score(yv, oof):.4f}  (n={len(idx)})")

base_mean = float(np.mean([r2_score(y_tr[tt_tr == tt], p14_oof[tt_tr == tt])
                           for tt in TARGETS]))
print(f"\nP14 mean R2 (equal-weight): {base_mean:.4f}")

# ---------- Physics laws (fit on train pairs) ----------
print("\n== physics ==", flush=True)
C, D = 1.0, 0.0
ok = np.isfinite(sib_tr[:, GI["egc"]]) & np.isfinite(sib_tr[:, GI["egb"]])
if int(ok.sum()) >= 3:
    C, D = np.polyfit(sib_tr[ok, GI["egc"]], sib_tr[ok, GI["egb"]], 1)
A2, B2 = 1.0, 0.0
ok = np.isfinite(sib_tr[:, GI["nc"]]) & np.isfinite(sib_tr[:, GI["eps"]])
if int(ok.sum()) >= 3:
    A2, B2 = np.polyfit(sib_tr[ok, GI["nc"]] ** 2, sib_tr[ok, GI["eps"]], 1)
n_pair_egb = int(ok.sum())
print(f"  egb = {C:.3f}*egc + {D:.3f}   (n_pairs={n_pair_egb})")
print(f"  eps = {A2:.4f}*nc^2 + {B2:.4f}   (n_pairs={int(ok.sum())})")

def phys_col(sc, kind):
    v = np.full(len(sc), np.nan, dtype=np.float64)
    if kind == "egb":
        src = sc[:, GI["egc"]]
        m = np.isfinite(src)
        v[m] = C * src[m] + D
    elif kind == "eps":
        src = sc[:, GI["nc"]]
        m = np.isfinite(src)
        v[m] = A2 * src[m] ** 2 + B2
    elif kind == "egc":
        ei = sc[:, GI["ei"]]
        ea = sc[:, GI["eea"]]
        m = np.isfinite(ei) & np.isfinite(ea)
        v[m] = ei[m] - ea[m]
    return v

phys_tr = {k: phys_col(sib_tr, k) for k in ("egb", "eps", "egc")}
phys_te = {k: phys_col(sib_te, k) for k in ("egb", "eps", "egc")}

# ---------- Final gated per-target meta blend ----------
print("\n== gated meta blend (alphas capped <= 0.30) ==")
final_oof = p14_oof.copy()
final_test = p14_test.copy()
report_rows = []
for tt in TARGETS:
    idx = np.where(tt_tr == tt)[0]
    j = GI[tt]
    keep = [k for k in range(7) if k != j]
    X = sib_tr[idx][:, keep]
    yv = y_tr[idx]
    gv = groups_tr[idx]
    n_g = len(np.unique(gv))
    n_s = min(FOLDS, max(2, n_g))
    cv = list(GroupKFold(n_splits=n_s).split(X, yv, gv))
    oar = oof_sib_ridge(X, yv, gv)
    par = np.full(len(idx), np.nan, dtype=np.float64)
    if tt in phys_tr:
        par = phys_tr[tt][idx]

    res_oof = np.zeros(len(idx))
    a_sibs, a_physs = [], []
    sib_cov = 0
    for tri, vai in cv:
        a_s = pick_alpha(yv[tri], p14_oof[idx][tri], oar[tri], ALPHA_SIB)
        a_p = pick_alpha(yv[tri], p14_oof[idx][tri], par[tri], ALPHA_PHYS)
        a_sibs.append(a_s)
        a_physs.append(a_p)
        v = p14_oof[idx][vai].copy()
        ok_s = np.isfinite(oar[vai])
        sib_cov += int(ok_s.sum())
        v[ok_s] = (1 - a_s) * v[ok_s] + a_s * oar[vai][ok_s]
        ok_p = np.isfinite(par[vai])
        v[ok_p] = (1 - a_p) * v[ok_p] + a_p * par[vai][ok_p]
        res_oof[vai] = v
    final_oof[idx] = res_oof

    a_s = float(np.mean(a_sibs))
    a_p = float(np.mean(a_physs))
    cm = np.nanmean(X, axis=0)
    cm = np.where(np.isfinite(cm), cm, 0.0)
    Xf = np.where(np.isfinite(X), X, cm)
    mdl = Ridge(alpha=1.0).fit(Xf, yv)
    idx_te = np.where(tt_te == tt)[0]
    f = p14_test[idx_te].copy()
    if len(idx_te):
        Xte = sib_te[idx_te][:, keep]
        okte = np.isfinite(Xte).sum(1) >= 1
        if okte.sum():
            sib_te_pred = mdl.predict(np.where(np.isfinite(Xte[okte]), Xte[okte], cm))
            f[okte] = (1 - a_s) * f[okte] + a_s * sib_te_pred
        if tt in phys_te:
            pte = phys_te[tt][idx_te]
            m2 = np.isfinite(pte)
            f[m2] = (1 - a_p) * f[m2] + a_p * pte[m2]
    final_test[idx_te] = f

    r_old = r2_score(yv, p14_oof[idx])
    r_new = r2_score(yv, final_oof[idx])
    report_rows.append((tt, r_old, r_new, r_new - r_old, a_s, a_p, sib_cov))
    print(f"  {tt}: a_sib={a_s:.3f} a_phys={a_p:.2f}  "
          f"P14={r_old:.4f}  CT={r_new:.4f}  d={r_new - r_old:+.4f}")

df = pd.DataFrame(report_rows,
                  columns=["target", "p14", "ct", "d", "a_sib", "a_phys", "sib_cov"])
new_mean = float(np.mean(df["ct"]))
deltas = df["ct"] - df["p14"]
sf = df[df["target"].isin(SMALL_FIVE)]
sf_w = float(np.mean(sf["ct"] - sf["p14"]))
worst = float(deltas.min())
alpha_max = float(max(df["a_sib"].max(), df["a_phys"].max()))

print("\n=== GATE REPORT ===")
print(df.round(4).to_string(index=False))
print(f"\nP14 mean          = {base_mean:.4f}")
print(f"CT mean           = {df['ct'].mean():.4f}")
print(f"mean delta        = {df['ct'].mean() - base_mean:+.4f}")
print(f"small-five mean d = {sf_w:+.4f} (need >= {GATE_GAIN})")
print(f"worst delta       = {worst:+.4f} (need > {GATE_REGRESS})")
print(f"max alpha         = {alpha_max:.2f} (need <= {CAP})")

gate_ok = (sf_w >= GATE_GAIN) and (worst > GATE_REGRESS) and (alpha_max <= CAP)
if gate_ok:
    sub = pd.DataFrame({"id": te["id"].values, "target": final_test})
    sub = sub.sort_values("id").reset_index(drop=True); sub.to_csv(os.path.join(OUT, "submission_ctpgcn.csv"), index=False)
    print(f"\nGATE PASS -> wrote submission_ctpgcn.csv (n={len(sub)}) -> ship")
else:
    print("\nGATE FAIL -> freeze P14 anchor (LB 0.883); no submission emitted")

